In [34]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBRegressor
import numpy as np
import optuna
import pandas as pd
import pickle
import warnings
warnings.filterwarnings("ignore")



SEED = 42
np.random.seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [35]:
df = pd.read_csv("recommender_model_dataset.csv")

print(df.shape)
display(df.head())

(5643, 49)


,player_id,season,alter,avg_rating_current,einsatzquote,tore_pro_spiel,tore_abs,assists_pro_spiel,assists_abs,rote_karten_pro_spiel,...,position_Linksaußen,position_Mittelfeld,position_Mittelstürmer,position_Offensives Mittelfeld,position_Rechter Verteidiger,position_Rechtes Mittelfeld,position_Rechtsaußen,position_Sturm,position_Torwart,position_Zentrales Mittelfeld
0,2866,2020,37.0,6.800000,0.769231,0.000000,0,0.000000,0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,3391,2020,34.0,7.233333,0.230769,0.333333,1,0.000000,0,0.0,...,0,0,0,1,0,0,0,0,0,0
2,4779,2021,39.0,6.850000,0.923077,0.083333,2,0.041667,1,0.0,...,0,0,0,1,0,0,0,0,0,0
3,10058,2020,35.0,7.041667,0.800000,0.000000,0,0.000000,0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,10058,2021,36.0,6.953333,1.000000,0.000000,0,0.000000,0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [36]:
print("Saisons:")
print(df["season"].value_counts().sort_index())

print("\nFehlende Werte:")
display(df.isna().mean().sort_values(ascending=False).head(20))

print("\nTarget Summary:")
print(df["ziel_rating_avg_naechste_saison"].describe())

print("\nPositionsspalten:")
position_columns = [c for c in df.columns if c.startswith("position_")]
print(position_columns)

Saisons:
season
2020     987
2021    1047
2022    1214
2023    1245
2024    1150
Name: count, dtype: int64

Fehlende Werte:


alter                              0.000709
player_id                          0.000000
position_Linker Verteidiger        0.000000
siege_abs                          0.000000
unentschieden_abs                  0.000000
niederlagen_abs                    0.000000
prozent_1_liga                     0.000000
prozent_pl                         0.000000
ziel_rating_avg_naechste_saison    0.000000
position_Abwehr                    0.000000
position_Defensives Mittelfeld     0.000000
position_Hängende Spitze           0.000000
position_Innenverteidiger          0.000000
position_Linkes Mittelfeld         0.000000
unentschieden_pro_spiel            0.000000
position_Linksaußen                0.000000
position_Mittelfeld                0.000000
position_Mittelstürmer             0.000000
position_Offensives Mittelfeld     0.000000
position_Rechter Verteidiger       0.000000
dtype: float64


Target Summary:
count    5643.000000
mean        6.846437
std         0.229117
min         3.700000
25%         6.700000
50%         6.832143
75%         6.976000
max         8.400000
Name: ziel_rating_avg_naechste_saison, dtype: float64

Positionsspalten:
['position_Abwehr', 'position_Defensives Mittelfeld', 'position_Hängende Spitze', 'position_Innenverteidiger', 'position_Linker Verteidiger', 'position_Linkes Mittelfeld', 'position_Linksaußen', 'position_Mittelfeld', 'position_Mittelstürmer', 'position_Offensives Mittelfeld', 'position_Rechter Verteidiger', 'position_Rechtes Mittelfeld', 'position_Rechtsaußen', 'position_Sturm', 'position_Torwart', 'position_Zentrales Mittelfeld']


In [37]:
POSITION_GROUPS = {
    "goalkeeper": [
        "position_Torwart"
    ],
    "defense": [
        "position_Linker Verteidiger",
        "position_Abwehr",
        "position_Rechter Verteidiger",
        "position_Innenverteidiger"
    ],
    "midfield": [
        "position_Defensives Mittelfeld",
        "position_Linkes Mittelfeld",
        "position_Mittelfeld",
        "position_Offensives Mittelfeld",
        "position_Zentrales Mittelfeld",
        "position_Rechtes Mittelfeld"
    ],
    "offense": [
        "position_Hängende Spitze",
        "position_Linksaußen",
        "position_Mittelstürmer",
        "position_Rechtsaußen",
        "position_Sturm"
    ]
}

all_position_cols = [c for c in df.columns if c.startswith("position_")]

for group_name, cols in POSITION_GROUPS.items():
    missing = [c for c in cols if c not in df.columns]
    print(f"{group_name}:")
    if len(missing) == 0:
        print("  alle Spalten vorhanden")
    else:
        print("  FEHLT:", missing)

goalkeeper:
  alle Spalten vorhanden
defense:
  alle Spalten vorhanden
midfield:
  alle Spalten vorhanden
offense:
  alle Spalten vorhanden


In [38]:
TARGET_COL = "ziel_rating_avg_naechste_saison"

def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2


def fit_position_model(
    df,
    group_name,
    position_cols,
    target_col=TARGET_COL,
    test_size=0.2,
    inner_valid_size=0.2,
    random_state=SEED,
    n_trials=40,
    early_stopping_rounds=50
):
    missing = [c for c in position_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Fehlende Positionsspalten für {group_name}: {missing}")

    subset = df[df[position_cols].sum(axis=1) > 0].copy()

    if subset.empty:
        raise ValueError(f"Keine Daten für Gruppe {group_name}")

    drop_other_position_cols = [c for c in all_position_cols if c not in position_cols]

    X = subset.drop(columns=[target_col, "player_id"] + drop_other_position_cols).copy()
    y = subset[target_col].copy()
    groups = subset["player_id"].copy()

    gss_outer = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=random_state
    )

    train_idx, test_idx = next(gss_outer.split(X, y, groups=groups))

    X_train_outer = X.iloc[train_idx].copy()
    y_train_outer = y.iloc[train_idx].copy()
    groups_train_outer = groups.iloc[train_idx].copy()

    X_test = X.iloc[test_idx].copy()
    y_test = y.iloc[test_idx].copy()
    groups_test = groups.iloc[test_idx].copy()

    overlap_outer = set(groups_train_outer.unique()).intersection(set(groups_test.unique()))
    if len(overlap_outer) > 0:
        raise ValueError(f"Spieler-Overlap zwischen Train und Test in {group_name}")

    baseline = DummyRegressor(strategy="mean")
    baseline.fit(X_train_outer, y_train_outer)
    baseline_preds = baseline.predict(X_test)
    baseline_mae, baseline_rmse, baseline_r2 = evaluate_regression(y_test, baseline_preds)

    gss_inner = GroupShuffleSplit(
        n_splits=1,
        test_size=inner_valid_size,
        random_state=random_state
    )

    inner_train_idx, inner_valid_idx = next(
        gss_inner.split(X_train_outer, y_train_outer, groups=groups_train_outer)
    )

    X_train_inner = X_train_outer.iloc[inner_train_idx].copy()
    y_train_inner = y_train_outer.iloc[inner_train_idx].copy()
    groups_train_inner = groups_train_outer.iloc[inner_train_idx].copy()

    X_valid_inner = X_train_outer.iloc[inner_valid_idx].copy()
    y_valid_inner = y_train_outer.iloc[inner_valid_idx].copy()
    groups_valid_inner = groups_train_outer.iloc[inner_valid_idx].copy()

    overlap_inner = set(groups_train_inner.unique()).intersection(set(groups_valid_inner.unique()))
    if len(overlap_inner) > 0:
        raise ValueError(f"Spieler-Overlap zwischen inner train und valid in {group_name}")

    def objective(trial):
        params = {
            "objective": "reg:squarederror",
            "random_state": random_state,
            "n_jobs": -1,
            "tree_method": "hist",
            "eval_metric": "mae",

            "n_estimators": trial.suggest_int("n_estimators", 200, 2000),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
        }

        model = XGBRegressor(
            **params,
            early_stopping_rounds=early_stopping_rounds
        )

        model.fit(
            X_train_inner,
            y_train_inner,
            eval_set=[(X_valid_inner, y_valid_inner)],
            verbose=False
        )

        valid_preds = model.predict(X_valid_inner)
        valid_mae = mean_absolute_error(y_valid_inner, valid_preds)

        return valid_mae

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=random_state),
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=5,
            n_warmup_steps=10
        )
    )

    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params = study.best_trial.params.copy()
    å
    best_model = XGBRegressor(
        objective="reg:squarederror",
        random_state=random_state,
        n_jobs=-1,
        tree_method="hist",
        eval_metric="mae",
        early_stopping_rounds=early_stopping_rounds,
        **best_params
    )

    best_model.fit(
        X_train_inner,
        y_train_inner,
        eval_set=[(X_valid_inner, y_valid_inner)],
        verbose=False
    )

    test_preds = best_model.predict(X_test)

    xgb_mae, xgb_rmse, xgb_r2 = evaluate_regression(y_test, test_preds)

    result_row = {
        "group": group_name,
        "n_rows": len(subset),
        "n_features": X.shape[1],
        "train_players": groups_train_outer.nunique(),
        "test_players": groups_test.nunique(),
        "baseline_mae": baseline_mae,
        "baseline_rmse": baseline_rmse,
        "baseline_r2": baseline_r2,
        "xgb_mae": xgb_mae,
        "xgb_rmse": xgb_rmse,
        "xgb_r2": xgb_r2,
        "mae_improvement_abs": baseline_mae - xgb_mae,
        "mae_improvement_pct": ((baseline_mae - xgb_mae) / baseline_mae) * 100 if baseline_mae != 0 else np.nan,
        "best_trial_value_mae": study.best_value,
        "best_iteration": getattr(best_model, "best_iteration", None),
        "best_score": getattr(best_model, "best_score", None),
        "best_params": best_params
    }

    pred_df = subset.iloc[test_idx][["player_id", "season"] + position_cols].copy()
    pred_df["y_true"] = y_test.values
    pred_df["y_pred"] = test_preds
    pred_df["abs_error"] = np.abs(pred_df["y_true"] - pred_df["y_pred"])
    pred_df["group"] = group_name

    feature_importance = pd.DataFrame({
        "feature": X_train_outer.columns,
        "importance": best_model.feature_importances_
    }).sort_values("importance", ascending=False)

    return {
        "group_name": group_name,
        "subset": subset,
        "X_train_outer": X_train_outer,
        "X_test": X_test,
        "y_train_outer": y_train_outer,
        "y_test": y_test,
        "groups_train_outer": groups_train_outer,
        "groups_test": groups_test,
        "X_train_inner": X_train_inner,
        "X_valid_inner": X_valid_inner,
        "y_train_inner": y_train_inner,
        "y_valid_inner": y_valid_inner,
        "baseline_model": baseline,
        "best_model": best_model,
        "study": study,
        "pred_df": pred_df,
        "feature_importance": feature_importance,
        "result_row": result_row
    }

In [39]:
group_overview = []

for group_name, cols in POSITION_GROUPS.items():
    available_cols = [c for c in cols if c in df.columns]
    subset = df[df[available_cols].sum(axis=1) > 0].copy()

    group_overview.append({
        "group": group_name,
        "rows": len(subset),
        "players": subset["player_id"].nunique()
    })

group_overview_df = pd.DataFrame(group_overview)
display(group_overview_df)

,group,rows,players
0,goalkeeper,459,220
1,defense,1981,864
2,midfield,1879,855
3,offense,1324,629


In [40]:
model_outputs = {}
results = []

for group_name, position_cols in POSITION_GROUPS.items():
    print(f"\n--- Training group: {group_name} ---")

    output = fit_position_model(
        df=df,
        group_name=group_name,
        position_cols=position_cols,
        target_col=TARGET_COL,
        test_size=0.2,
        inner_valid_size=0.2,
        random_state=SEED,
        n_trials=40,
        early_stopping_rounds=50
    )

    model_outputs[group_name] = output
    results.append(output["result_row"])


--- Training group: goalkeeper ---

--- Training group: defense ---

--- Training group: midfield ---

--- Training group: offense ---


In [41]:
results_df = pd.DataFrame(results)

display(
    results_df[
        [
            "group",
            "n_rows",
            "train_players",
            "test_players",
            "baseline_mae",
            "xgb_mae",
            "baseline_rmse",
            "xgb_rmse",
            "baseline_r2",
            "xgb_r2",
            "mae_improvement_pct"
        ]
    ].sort_values("xgb_mae")
)

,group,n_rows,train_players,test_players,baseline_mae,xgb_mae,baseline_rmse,xgb_rmse,baseline_r2,xgb_r2,mae_improvement_pct
1,defense,1981,691,173,0.146217,0.138849,0.213731,0.204094,-0.006425,0.082288,5.038866
2,midfield,1879,684,171,0.155423,0.141178,0.212042,0.196193,-0.005225,0.139426,9.165567
0,goalkeeper,459,176,44,0.143513,0.145491,0.221970,0.222440,-0.002441,-0.006697,-1.378250
3,offense,1324,503,126,0.195521,0.169883,0.252899,0.225464,-0.015622,0.192784,13.112708


In [42]:
for group_name, output in model_outputs.items():
    print(f"\n### {group_name}")
    print("Best trial value (MAE):", output["study"].best_value)
    print("Best params:", output["study"].best_trial.params)
    print("Best iteration:", output["result_row"]["best_iteration"])
    print("Best score:", output["result_row"]["best_score"])


### goalkeeper
Best trial value (MAE): 0.14982169742367415
Best params: {'n_estimators': 1510, 'max_depth': 8, 'learning_rate': 0.008875718438156295, 'subsample': 0.7446352093828248, 'colsample_bytree': 0.848706822701965, 'min_child_weight': 9, 'gamma': 0.0124954514386314, 'reg_alpha': 0.010351134098833557, 'reg_lambda': 0.003275246668615685}
Best iteration: 124
Best score: 0.14982170740763345

### defense
Best trial value (MAE): 0.1253275551827105
Best params: {'n_estimators': 568, 'max_depth': 3, 'learning_rate': 0.010625811792432868, 'subsample': 0.6004749960247624, 'colsample_bytree': 0.8558129624201825, 'min_child_weight': 10, 'gamma': 0.017543601070423097, 'reg_alpha': 0.018607074770103676, 'reg_lambda': 0.0017800963989018325}
Best iteration: 208
Best score: 0.12532756313587884

### midfield
Best trial value (MAE): 0.12989019756796924
Best params: {'n_estimators': 568, 'max_depth': 3, 'learning_rate': 0.010625811792432868, 'subsample': 0.6004749960247624, 'colsample_bytree': 0.8

In [43]:
for group_name, output in model_outputs.items():
    print(f"\n### Feature Importance: {group_name}")
    display(output["feature_importance"].head(15))


### Feature Importance: goalkeeper


,feature,importance
3,einsatzquote,0.071888
20,team_tore_abs,0.068809
25,niederlagen_pro_spiel,0.068735
2,avg_rating_current,0.067107
17,minuten_abs,0.067094
16,minuten_pro_spiel,0.060833
21,team_gegentore_pro_spiel,0.059309
22,team_gegentore_abs,0.058771
18,anzahl_spiele,0.056570
1,alter,0.054457



### Feature Importance: defense


,feature,importance
21,team_gegentore_pro_spiel,0.081166
16,minuten_pro_spiel,0.070006
19,team_tore_pro_spiel,0.062277
17,minuten_abs,0.061753
1,alter,0.050650
2,avg_rating_current,0.050646
20,team_tore_abs,0.050350
30,prozent_pl,0.047427
25,niederlagen_pro_spiel,0.045230
22,team_gegentore_abs,0.043503



### Feature Importance: midfield


,feature,importance
2,avg_rating_current,0.112393
16,minuten_pro_spiel,0.074876
4,tore_pro_spiel,0.053713
35,position_Rechtes Mittelfeld,0.047642
5,tore_abs,0.043827
7,assists_abs,0.037882
19,team_tore_pro_spiel,0.037006
17,minuten_abs,0.036459
30,prozent_pl,0.036363
32,position_Linkes Mittelfeld,0.036300



### Feature Importance: offense


,feature,importance
20,team_tore_abs,0.150867
2,avg_rating_current,0.115590
21,team_gegentore_pro_spiel,0.068556
4,tore_pro_spiel,0.062472
17,minuten_abs,0.047265
22,team_gegentore_abs,0.046009
19,team_tore_pro_spiel,0.043830
0,season,0.043145
25,niederlagen_pro_spiel,0.042856
30,prozent_pl,0.042649


In [44]:
for group_name, output in model_outputs.items():
    print(f"\n### Predictions: {group_name}")
    display(output["pred_df"].sort_values("abs_error").head(10))
    display(output["pred_df"].sort_values("abs_error", ascending=False).head(10))


### Predictions: goalkeeper


,player_id,season,position_Torwart,y_true,y_pred,abs_error,group
2817,521437,2020,1,7.058333,7.057550,0.000783,goalkeeper
3118,578924,2021,1,7.100000,7.096476,0.003524,goalkeeper
2702,507449,2023,1,7.097059,7.103621,0.006562,goalkeeper
2004,395671,2021,1,7.100000,7.092395,0.007605,goalkeeper
521,183308,2020,1,7.062500,7.071894,0.009394,goalkeeper
619,195340,2020,1,7.157692,7.148224,0.009468,goalkeeper
3937,685581,2020,1,7.041176,7.029163,0.012014,goalkeeper
555,189066,2020,1,7.104545,7.116985,0.012439,goalkeeper
4722,874337,2024,1,7.100000,7.087378,0.012622,goalkeeper
1892,388212,2022,1,7.100000,7.113776,0.013776,goalkeeper


,player_id,season,position_Torwart,y_true,y_pred,abs_error,group
5095,932195,2024,1,8.1000,7.052410,1.047590,goalkeeper
3936,685434,2022,1,6.3000,7.071919,0.771919,goalkeeper
5553,1206410,2024,1,7.7000,7.052030,0.647970,goalkeeper
4987,926177,2023,1,6.5000,7.017737,0.517737,goalkeeper
2527,499195,2022,1,6.7000,7.124104,0.424104,goalkeeper
932,248823,2021,1,6.6500,7.070664,0.420664,goalkeeper
2124,421783,2022,1,6.6000,7.019270,0.419270,goalkeeper
931,248823,2020,1,6.8000,7.150556,0.350556,goalkeeper
3940,685581,2023,1,7.4500,7.106060,0.343940,goalkeeper
4605,819307,2024,1,7.2875,6.980215,0.307285,goalkeeper



### Predictions: defense


,player_id,season,position_Linker Verteidiger,position_Abwehr,position_Rechter Verteidiger,position_Innenverteidiger,y_true,y_pred,abs_error,group
3319,604822,2023,0,0,1,0,6.748000,6.746985,0.001015,defense
5083,932162,2023,0,0,0,1,6.800000,6.803006,0.003006,defense
2111,418656,2024,0,0,1,0,6.794444,6.797611,0.003167,defense
3694,639703,2021,0,0,0,1,6.883333,6.880042,0.003292,defense
2018,401039,2023,0,0,1,0,6.800000,6.804235,0.004235,defense
2019,401039,2024,0,0,1,0,6.815789,6.810843,0.004946,defense
4476,805766,2022,0,0,0,1,6.822727,6.827946,0.005218,defense
2723,507478,2024,1,0,0,0,6.846667,6.852203,0.005537,defense
3857,677432,2022,0,0,1,0,6.716667,6.710447,0.006220,defense
5045,927420,2023,1,0,0,0,6.782759,6.776447,0.006312,defense


...

,player_id,season,position_Defensives Mittelfeld,position_Linkes Mittelfeld,position_Mittelfeld,position_Offensives Mittelfeld,position_Zentrales Mittelfeld,position_Rechtes Mittelfeld,y_true,y_pred,abs_error,group
3994,691358,2021,1,0,0,0,0,0,6.852000,6.853324,0.001324,midfield
949,252072,2022,0,1,0,0,0,0,6.700000,6.697647,0.002353,midfield
5021,927395,2022,0,1,0,0,0,0,6.680000,6.682547,0.002547,midfield
2121,419659,2024,0,0,0,0,1,0,6.975000,6.971667,0.003333,midfield
5447,1135115,2023,0,0,0,1,0,0,6.875000,6.878341,0.003341,midfield
3232,601544,2024,0,0,0,0,0,1,6.864286,6.867772,0.003486,midfield
4177,707626,2022,0,0,0,0,1,0,6.909091,6.912796,0.003705,midfield
418,165511,2023,0,0,0,0,1,0,6.882609,6.876463,0.006146,midfield
4679,864371,2023,0,0,0,0,1,0,6.863636,6.857298,0.006338,midfield
1516,342097,2021,1,0,0,0,0,0,6.886667,6.880012,0.006655,midfield


,player_id,season,position_Defensives Mittelfeld,position_Linkes Mittelfeld,position_Mittelfeld,position_Offensives Mittelfeld,position_Zentrales Mittelfeld,position_Rechtes Mittelfeld,y_true,y_pred,abs_error,group
2616,500771,2020,0,0,0,1,0,0,8.400000,6.854001,1.545999,midfield
1500,341322,2024,0,0,0,0,1,0,7.673684,6.997953,0.675731,midfield
2685,507412,2020,0,0,0,1,0,0,7.533333,6.925848,0.607485,midfield
2549,499212,2023,1,0,0,0,0,0,6.325000,6.878102,0.553102,midfield
2678,507407,2021,0,0,0,1,0,0,7.446667,6.919203,0.527463,midfield
1993,394445,2020,0,1,0,0,0,0,7.265000,6.763132,0.501868,midfield
2120,419659,2023,0,0,0,0,1,0,7.208000,6.726524,0.481476,midfield
3780,659399,2024,0,0,0,0,0,1,7.462500,6.987421,0.475079,midfield
1496,341322,2020,0,0,0,0,1,0,7.500000,7.029415,0.470585,midfield
3758,659352,2024,0,0,0,0,1,0,7.211111,6.761539,0.449572,midfield



### Predictions: offense


,player_id,season,position_Hängende Spitze,position_Linksaußen,position_Mittelstürmer,position_Rechtsaußen,position_Sturm,y_true,y_pred,abs_error,group
3844,675622,2024,0,0,1,0,0,6.800000,6.800269,0.000269,offense
3234,601613,2021,0,0,1,0,0,6.733333,6.734151,0.000818,offense
551,187597,2020,0,0,1,0,0,6.845833,6.844738,0.001096,offense
5580,1245119,2023,0,0,0,1,0,6.807692,6.808961,0.001269,offense
4236,711192,2020,0,0,1,0,0,6.664706,6.666019,0.001313,offense
2745,507499,2024,0,0,1,0,0,6.692308,6.690716,0.001592,offense
2869,524864,2022,0,0,0,1,0,6.900000,6.897952,0.002048,offense
2164,426889,2021,0,0,1,0,0,6.660000,6.657923,0.002077,offense
4983,925929,2024,0,0,0,1,0,6.850000,6.852293,0.002293,offense
4947,923834,2023,0,0,1,0,0,6.838235,6.833171,0.005064,offense


,player_id,season,position_Hängende Spitze,position_Linksaußen,position_Mittelstürmer,position_Rechtsaußen,position_Sturm,y_true,y_pred,abs_error,group
1000,260163,2024,0,1,0,0,0,7.915789,6.983675,0.932114,offense
4609,823521,2023,0,0,1,0,0,7.533333,6.685391,0.847942,offense
1369,322512,2024,0,0,1,0,0,7.673684,6.874158,0.799526,offense
3493,620795,2020,0,1,0,0,0,7.553571,6.857014,0.696557,offense
4712,870270,2022,0,0,1,0,0,7.300000,6.741567,0.558433,offense
3692,638571,2021,0,0,1,0,0,7.500000,6.944033,0.555967,offense
4714,870270,2024,0,0,1,0,0,7.458824,6.904306,0.554518,offense
2953,534840,2021,0,0,1,0,0,7.375862,6.875391,0.500471,offense
552,187597,2021,0,0,1,0,0,7.308333,6.822030,0.486304,offense
2170,427125,2022,0,0,0,1,0,7.244444,6.770829,0.473616,offense


In [45]:
for group_name, output in model_outputs.items():
    file_name = f"xgb_{group_name}_model.pkl"
    with open(file_name, "wb") as f:
        pickle.dump(output["best_model"], f)
    print(f"Saved as: {file_name}")

Saved as: xgb_goalkeeper_model.pkl
Saved as: xgb_defense_model.pkl
Saved as: xgb_midfield_model.pkl
Saved as: xgb_offense_model.pkl


In [46]:
summary_table = results_df[
    ["group", "baseline_mae", "xgb_mae", "baseline_r2", "xgb_r2"]
].copy()

summary_table = summary_table.rename(columns={
    "group": "Position",
    "baseline_mae": "Baseline MAE",
    "xgb_mae": "XGBoost MAE",
    "baseline_r2": "Baseline R2",
    "xgb_r2": "XGBoost R2"
})

summary_table["Position"] = summary_table["Position"].replace({
    "goalkeeper": "Goalkeeper",
    "defense": "Defense",
    "midfield": "Midfield",
    "offense": "Offense"
})

summary_table = summary_table.sort_values("Position").reset_index(drop=True)

display(summary_table.round(4))

,Position,Baseline MAE,XGBoost MAE,Baseline R2,XGBoost R2
0,Defense,0.1462,0.1388,-0.0064,0.0823
1,Goalkeeper,0.1435,0.1455,-0.0024,-0.0067
2,Midfield,0.1554,0.1412,-0.0052,0.1394
3,Offense,0.1955,0.1699,-0.0156,0.1928
